# 0 — Prerequisites: image processing + QuPath segmentation

This pipeline starts from **QuPath segmentation outputs**, which are themselves
downstream of raw-image processing. The two upstream stages are *not* part of this
repo:

1. **Image processing** (illumination correction, stitching, deconvolution, EDF,
   registration, autofluorescence removal) — use the author's
   [KINTSUGI](https://github.com/smith6jt-cop/KINTSUGI) pipeline (STAR Protocols 2025).
2. **Cell segmentation** in [QuPath](https://qupath.github.io/) (InstanSeg / StarDist),
   then export **two artifacts** this pipeline consumes:
   - the per-cell **measurement CSV** (`Cellmeasurements.csv`), and
   - full-resolution per-cell **GeoJSON** boundaries (feature `id` == detection UUID).

Set the paths to these in `../config.ini` (`[paths] images_dir`, `cells_csv`,
`donor_metadata`).

In [ ]:
# Resolve the pipeline configuration (paths + params from ../config.ini).
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # make `phenocycler` importable from notebooks/
from phenocycler import load_config
cfg = load_config(pathlib.Path.cwd().parent / 'config.ini')
print('data_dir     :', cfg.data_dir)
print('images_dir   :', cfg.images_dir)
print('cells_csv    :', cfg.cells_csv)
print('n_jobs       :', cfg.n_jobs, '| use_gpu:', cfg.use_gpu)
cfg.discover_donors()  # donors found under data/cells/donor_id=* (empty until Step 1)

## Export per-cell GeoJSON (REDSEA input)

Run the QuPath Groovy exporter **once per image** (GUI closed). It writes
`data/redsea_scratch/geojson/cells__<image>.geojson`, which `redsea` rasterizes.

In [ ]:
# Shown for reference — run these in a shell with QuPath installed, not in the kernel.
print(r'''
# single image (headless):
QuPath script --project <project.qpproj> --image "<image name>" \\
    scripts/groovy/export_cells_geojson.groovy

# or batch every image missing a GeoJSON:
bash scripts/senior_export_new_geojsons.sh   # (adapt paths for your project)
''')
print('geojson dir:', cfg.geojson_dir)